# Statistics for ML Monitoring: Detecting Data Drift with Hypothesis Tests

**Dataset:** `sklearn.datasets.load_breast_cancer` — real diagnostic measurements (cell nuclei features)
from breast mass biopsies, labelled malignant/benign. Bundled with scikit-learn, no download required.

**Problem statement.** Imagine this dataset's features (`mean radius`, `mean texture`, etc.) are the
inputs to a deployed tumor-classification model. As SQA-turned-ML-quality engineer, your job isn't to
retrain the model — it's to build the **statistical test suite** that watches incoming batches of data
and flags when a feature's distribution has drifted away from what the model was trained on. A silent
drift (e.g. a new clinic sends a batch skewed toward more aggressive tumors, or a lab recalibrates an
instrument) is exactly the kind of regression that unit tests on code can't catch, but a statistical
test on data *can*.

You will build a two-sample Kolmogorov–Smirnov (KS) drift detector, apply a multiple-testing correction
(Bonferroni) so that testing 30 features at once doesn't flood you with false alarms, and validate it
against three realistic scenarios:

- **Scenario A — no real drift:** two random samples drawn from the *same* population. A good detector
  should flag very few features here (controlling the false-positive rate is as important as catching
  real drift).
- **Scenario B — genuine real-world drift:** reference and production batches resampled to have
  different malignant/benign *prevalence* (e.g. a new clinic sees sicker patients). This is real data,
  not synthetic — the shift comes purely from changing the class mixture, which realistically shifts
  many feature distributions at once.
- **Scenario C — controlled injected drift:** one feature in the production batch gets a small constant
  shift of known size. This is the one place we touch real values directly, and only because we need a
  shift of *known, controllable magnitude* to sanity-check that the detector's sensitivity scales the
  way you'd expect — that's not something scenario B's natural resampling can guarantee.

**What you should produce:**
1. Three functions that build the reference/production DataFrame pairs for scenarios A, B, and C.
2. A function `ks_drift_report(reference, production, feature_cols, alpha=0.05)` that runs a KS test per
   feature, applies a Bonferroni-corrected significance threshold, and returns a tidy results DataFrame
   with columns `feature`, `ks_stat`, `p_value`, `bonferroni_threshold`, `flagged`.
3. A short comparison across the three scenarios: how many features get flagged in each, and whether
   that matches what you'd expect (few in A, many in B, exactly one — the injected one — reliably caught
   in C, including at a smaller injected shift size).


## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer

RNG_SEED = 0
np.random.seed(RNG_SEED)


In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
# target: 0 = malignant, 1 = benign (per sklearn's encoding)
df["target"] = data.target
feature_cols = list(data.feature_names)

print(df.shape)
print(df["target"].value_counts(normalize=True))
df.head()


### Step 1 — Build the three monitoring scenarios

Fill in each function below. All three should return a `(reference_df, production_df)` tuple, each a
DataFrame containing just `feature_cols` (drop `target` before running the detector — in production you
usually don't have ground-truth labels for incoming data yet, only features).

- `make_no_drift_scenario`: draw two disjoint random samples of size `n` each from the full dataset,
  ignoring `target` (simulates two production batches from the same stable population).
- `make_class_shift_scenario`: build `reference` with `benign_frac` of its rows benign (default should
  mimic the dataset's natural ~63% benign prevalence) and `production` with the prevalence flipped
  (mostly malignant) — same `n` rows each, sampled *with* replacement from each class pool so you can hit
  any requested `n` even though each class has limited rows.
- `make_injected_shift_scenario`: `reference` is a plain random sample of size `n`; `production` is a
  *different* random sample of size `n` with `shift` added to column `feature` only.


In [ ]:
def make_no_drift_scenario(df, feature_cols, n=150, seed=RNG_SEED):
    """
    TODO: sample 2*n rows without replacement, split into two disjoint
    DataFrames of n rows each (reference, production), both restricted
    to feature_cols. No distributional difference should exist between them
    beyond sampling noise.
    """
    raise NotImplementedError


def make_class_shift_scenario(df, feature_cols, n=150, ref_benign_frac=0.63,
                               prod_benign_frac=0.10, seed=RNG_SEED):
    """
    TODO: build `reference` by sampling n rows total from df, drawing
    ref_benign_frac of them (rounded) from the benign rows and the rest from
    malignant rows (sample with replacement within each class so n can exceed
    the class's row count). Build `production` the same way but using
    prod_benign_frac instead. Return (reference[feature_cols], production[feature_cols]).
    """
    raise NotImplementedError


def make_injected_shift_scenario(df, feature_cols, feature="mean radius",
                                  shift=2.0, n=150, seed=RNG_SEED):
    """
    TODO: draw two disjoint random samples of n rows each (reference,
    production) restricted to feature_cols, then add `shift` to
    production[feature] in place. Return (reference, production).
    """
    raise NotImplementedError


### Step 2 — Implement the KS-test drift detector

Implement `ks_drift_report`. For each column in `feature_cols`, run `scipy.stats.ks_2samp(reference[col],
production[col])` to get a KS statistic and p-value. Testing many features at once inflates your
false-positive rate (run 30 independent tests at `alpha=0.05` and you'd expect ~1-2 false alarms by
chance alone) — correct for this with a **Bonferroni correction**: divide `alpha` by the number of
features tested to get the per-feature significance threshold, and flag a feature only if its p-value
falls below that corrected threshold.


In [ ]:
def ks_drift_report(reference, production, feature_cols, alpha=0.05):
    """
    TODO: for each col in feature_cols, run stats.ks_2samp(reference[col], production[col]).
    Compute bonferroni_threshold = alpha / len(feature_cols).
    flagged = p_value < bonferroni_threshold.
    Return a DataFrame with columns:
        feature, ks_stat, p_value, bonferroni_threshold, flagged
    sorted by p_value ascending.
    """
    raise NotImplementedError


### Step 3 — Run the detector on all three scenarios

Build each scenario, run `ks_drift_report` on it, and print how many features were flagged. Also run
`make_injected_shift_scenario` at a couple of different `shift` sizes (e.g. 0.3 vs 2.0 standard-deviation
scale) to see how detection depends on effect size — note `mean radius` has roughly a 3.5 unit standard
deviation in this dataset, so pick shift values accordingly if you want to compare a "small" vs "large"
shift.


In [ ]:
# TODO:
# 1. ref_a, prod_a = make_no_drift_scenario(...); report_a = ks_drift_report(...)
# 2. ref_b, prod_b = make_class_shift_scenario(...); report_b = ks_drift_report(...)
# 3. ref_c, prod_c = make_injected_shift_scenario(...); report_c = ks_drift_report(...)
# 4. Print len(flagged) for each scenario and display the top few rows of each report.


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

```python
def make_no_drift_scenario(df, feature_cols, n=150, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    idx = rng.choice(df.index, size=2 * n, replace=False)
    ref_idx, prod_idx = idx[:n], idx[n:]
    reference = df.loc[ref_idx, feature_cols].reset_index(drop=True)
    production = df.loc[prod_idx, feature_cols].reset_index(drop=True)
    return reference, production


def make_class_shift_scenario(df, feature_cols, n=150, ref_benign_frac=0.63,
                               prod_benign_frac=0.10, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    benign = df[df["target"] == 1]
    malignant = df[df["target"] == 0]

    def sample_mix(benign_frac, tag):
        n_benign = round(n * benign_frac)
        n_malignant = n - n_benign
        b = benign.sample(n=n_benign, replace=True, random_state=rng.integers(1e6))
        m = malignant.sample(n=n_malignant, replace=True, random_state=rng.integers(1e6))
        return pd.concat([b, m], ignore_index=True)[feature_cols]

    reference = sample_mix(ref_benign_frac, "ref")
    production = sample_mix(prod_benign_frac, "prod")
    return reference, production


def make_injected_shift_scenario(df, feature_cols, feature="mean radius",
                                  shift=2.0, n=150, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    idx = rng.choice(df.index, size=2 * n, replace=False)
    ref_idx, prod_idx = idx[:n], idx[n:]
    reference = df.loc[ref_idx, feature_cols].reset_index(drop=True)
    production = df.loc[prod_idx, feature_cols].reset_index(drop=True)
    production[feature] = production[feature] + shift
    return reference, production


def ks_drift_report(reference, production, feature_cols, alpha=0.05):
    rows = []
    threshold = alpha / len(feature_cols)
    for col in feature_cols:
        ks_stat, p_value = stats.ks_2samp(reference[col], production[col])
        rows.append({
            "feature": col,
            "ks_stat": ks_stat,
            "p_value": p_value,
            "bonferroni_threshold": threshold,
            "flagged": p_value < threshold,
        })
    return pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)


# --- Scenario A: no real drift ---
ref_a, prod_a = make_no_drift_scenario(df, feature_cols, n=150)
report_a = ks_drift_report(ref_a, prod_a, feature_cols)
print("Scenario A (no drift) flagged:", report_a["flagged"].sum(), "/", len(feature_cols))

# --- Scenario B: genuine class-prevalence drift ---
ref_b, prod_b = make_class_shift_scenario(df, feature_cols, n=150)
report_b = ks_drift_report(ref_b, prod_b, feature_cols)
print("Scenario B (class-mix drift) flagged:", report_b["flagged"].sum(), "/", len(feature_cols))

# --- Scenario C: controlled injected shift, two magnitudes ---
ref_c_small, prod_c_small = make_injected_shift_scenario(df, feature_cols, shift=0.3, n=150)
report_c_small = ks_drift_report(ref_c_small, prod_c_small, feature_cols)
print("Scenario C (small injected shift) flagged:",
      report_c_small.loc[report_c_small["flagged"], "feature"].tolist())

ref_c_big, prod_c_big = make_injected_shift_scenario(df, feature_cols, shift=2.0, n=150)
report_c_big = ks_drift_report(ref_c_big, prod_c_big, feature_cols)
print("Scenario C (large injected shift) flagged:",
      report_c_big.loc[report_c_big["flagged"], "feature"].tolist())
```

**Interpretation.**

- **Scenario A** should flag close to zero features (occasionally one, by chance) — this is exactly what
  the Bonferroni correction buys you: without it, testing 30 features at `alpha=0.05` would flag ~1-2
  features *by chance alone* on every single batch, which would train everyone to ignore the alerts.
- **Scenario B** typically flags a large fraction of the 30 features. That's expected and correct: cell
  nuclei from malignant tumors differ systematically across most measured dimensions (radius, texture,
  concavity, …), so shifting the malignant/benign mix genuinely shifts most marginal distributions at
  once. A real "new clinic population" drift would look like this — many features move together, which
  is itself a diagnostic clue that the shift is in the label distribution, not instrument noise.
- **Scenario C (small shift, 0.3)** should flag `mean radius` rarely or not at all at `n=150` — a shift
  of less than 1/10th of a standard deviation is hard to detect at this sample size. **Scenario C (large
  shift, 2.0)**, which is more than half a standard deviation, should reliably flag exactly `mean radius`
  and nothing else, confirming the detector is both sensitive enough to catch a real shift and specific
  enough not to implicate unrelated features. This is the core lesson for ML quality work: a drift
  detector's usefulness depends on tuning sample size and significance thresholds to the effect sizes you
  actually care about catching — "no drift detected" is not the same as "no drift happened" if your
  batches are too small or your correction too conservative.

</details>
